# 09 — Experimento 3: resultados

Evaluación de las explicaciones del `08_experimento_3`: comprobaciones automáticas, tipos de error, lo que dice el modelo frente a tu referencia, puntuación con la rúbrica y **H₃**. **No necesita GPU.**

**H₃** (Y. Liu, Tao, Meng, Wang et al., 2024, rúbrica adaptada): en las **20 alertas de consenso**, media ≥ 4 en utilidad y en legibilidad, y al menos el 80 % de las explicaciones sin errores de fidelidad (ni citas equivocadas en la comprobación automática ni `error_fidelidad = 1` en la revisión manual).

| Sección | Qué hace |
|---|---|
| 1–3 | Entorno, ventanas y ejecuciones (las copiadas o hechas en el `08_`) |
| 4 | Tabla por ventana, con las comprobaciones recalculadas |
| 5 | Estadísticas: comprobaciones por grupo, tipos de error y diagnóstico frente a tu referencia |
| 6 | Gráficas |
| 7 | Resumen |
| 8 | Informe HTML con desplegable para revisar cada ventana (privado) |
| 9 | Puntuaciones y H₃ |
| 10 | Guardado de las cifras en `resultados/experimento_3/` |

**Grupos:** *alertas* = 20 alertas de consenso de prueba (las de H₃); *resto de prueba* = 10 de alta actividad sin alerta y 10 al azar; *desarrollo* = las 29 con las que se ajustó el prompt (sus cifras son optimistas).

**Confidencial:** el notebook solo muestra cifras y categorías. Las respuestas del modelo, los errores de cita con los nombres reales y tus comentarios están solo en `privado/`.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import glob
import json
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from config import (RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, RUTA_EXPERIMENTO_2_PRIVADO,
                    RUTA_EXPERIMENTO_3_PRIVADO, RUTA_EXPERIMENTO_3)
from llm_lab import exp2, exp3
from llm_lab import exp2_estadisticas as est
from llm_lab.huellas import huella_texto
from llm_lab.prompts import Prompts
from llm_lab.ventanas import FuenteVentanas

RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"
RUTA_EJECUCIONES = f"{RUTA_EXPERIMENTO_3_PRIVADO}/ejecuciones"
RUTA_INFORMES = f"{RUTA_EXPERIMENTO_3_PRIVADO}/informes"
RUTA_EVALUACION_PRIVADA = f"{RUTA_EXPERIMENTO_3_PRIVADO}/evaluacion"   # puntuaciones con tus comentarios

prompts = Prompts()
fuente = FuenteVentanas(RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP)
pio.renderers.default = "vscode"   # "colab" si lo ejecutas en Colab desde el navegador

VARIANTE = "exp3_calculos@v1"
HUELLA = huella_texto(prompts.texto_prompt(VARIANTE))
PREFIJO = "exp3_calc"
print("Huella del prompt:", HUELLA)

## 2. Ventanas

In [ ]:
# --- 2. Ventanas: alertas de consenso, resto de prueba y desarrollo (con la huella comprobada) ---
prueba = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_prueba.csv", fuente)
desarrollo = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/ventanas_desarrollo.csv", fuente)
VENTANAS = prueba + desarrollo
GRUPOS = ["alertas", "resto de prueba", "desarrollo"]
SUBCONJUNTO = {w["id"]: ("alertas" if w["grupo"] == "alerta_consenso_comun" else "resto de prueba") for w in prueba}
SUBCONJUNTO.update({w["id"]: "desarrollo" for w in desarrollo})
print({g: list(SUBCONJUNTO.values()).count(g) for g in GRUPOS})

## 3. Ejecuciones

Las de `privado/experimento_3/ejecuciones/` con el prompt de la 1 (misma huella). Por defecto, la más reciente con **las 40 de prueba** y la más reciente con **las 29 de desarrollo**; puedes fijar otras en `EJECUCION_PRUEBA` y `EJECUCION_DESARROLLO`.

In [ ]:
# --- 3. Ejecuciones con este prompt ---
EJECUCION_PRUEBA = None        # p. ej. "exp3_calc_qwen3.5-9b_20261013_181507"; None = la más reciente con las 40
EJECUCION_DESARROLLO = None    # p. ej. "exp3_calc_qwen3.5-9b_20261013_191237"; None = la más reciente con las 29

ids_prueba, ids_desarrollo = {w["id"] for w in prueba}, {w["id"] for w in desarrollo}
lista = []
for ruta in sorted(glob.glob(f"{RUTA_EJECUCIONES}/{PREFIJO}_*.json")):
    r = json.load(open(ruta, encoding="utf-8"))
    ids = {c["id"] for c in r["llamadas"]}
    lista.append({"ejecucion": os.path.basename(ruta)[:-5], "huella": r["huella_prompt"],
                  "prueba": len(ids & ids_prueba), "desarrollo": len(ids & ids_desarrollo)})
lista = pd.DataFrame(lista)
display(lista)
validas = lista[lista["huella"] == HUELLA]
EJECUCION_PRUEBA = EJECUCION_PRUEBA or validas[validas["prueba"] == len(ids_prueba)]["ejecucion"].iloc[-1]
EJECUCION_DESARROLLO = EJECUCION_DESARROLLO or validas[validas["desarrollo"] == len(ids_desarrollo)]["ejecucion"].iloc[-1]
print("Prueba:", EJECUCION_PRUEBA, "· desarrollo:", EJECUCION_DESARROLLO)

## 4. Tabla por ventana

Comprobaciones recalculadas con `llm_lab.exp3` a partir de las respuestas guardadas: citas (lee `evN: señal=valor`, `evN (señal=valor)` y `señal=valor en evN`; no lee los paréntesis con «respectivamente»), valores calculados, nivel de impacto frente a la escala y número de citas equivocadas de cada tipo. Aquí solo se muestran cifras y categorías; los errores de cita con los nombres de las señales están en el informe HTML (sección 8).

**Tipos de error de cita** (aproximación: con valores pequeños varias celdas pueden coincidir por azar): *reporte vecino* (el valor es el de esa señal en el reporte anterior o el siguiente), *otra señal, mismo reporte*, *otra señal, reporte vecino*, *valor de otra celda*, *no está en la ventana* y *señal o reporte inexistente*.

In [ ]:
# --- 4. Tabla por ventana (sin columnas de texto) ---
REGISTROS, partes = {}, []
for base, grupos in ((EJECUCION_PRUEBA, ("alertas", "resto de prueba")), (EJECUCION_DESARROLLO, ("desarrollo",))):
    REGISTROS[base], t = exp3.cargar_ejecucion(f"{RUTA_EJECUCIONES}/{base}.json", prompts, fuente, VENTANAS)
    t["subconjunto"] = t["id"].map(SUBCONJUNTO)
    partes.append(t[t["subconjunto"].isin(grupos)].assign(ejecucion=base))
tabla = pd.concat(partes, ignore_index=True)
tabla["subconjunto"] = pd.Categorical(tabla["subconjunto"], GRUPOS, ordered=True)
tabla = tabla.sort_values(["subconjunto", "id"]).reset_index(drop=True)
COLUMNAS_ERROR = [f"error: {t}" for t in exp3.TIPOS_ERROR]
VISIBLES = ["subconjunto", "id", "referencia", "estado_json", "citas", "citas_correctas", "calculos", "calculos_correctos",
            "origen", "evolucion_tipo", "impacto_video", "impacto_video_esperado", "impacto_audio", "impacto_audio_esperado"]
with pd.option_context("display.max_rows", 80, "display.max_columns", 40):
    display(tabla[VISIBLES])

## 5. Estadísticas

- **5.1** Comprobaciones automáticas por grupo (IC 95 %, Clopper-Pearson). El **nivel de impacto coherente con la escala** es una medida aparte de H₃: compara el nivel que da el modelo con el de la escala declarada (peor reporte de vídeo y de audio: 0 ninguno, 1–10 bajo, 11–360 medio, > 360 alto). Sus fallos apuntan a mejoras del prompt (jerarquía de la escala y cómo se pide aplicarla), no solo a límites del modelo.
- **5.2** Citas equivocadas por tipo de error y grupo.
- **5.3** Lo que dice el modelo frente a tu referencia: origen, evolución e impacto en vídeo y en audio.

In [ ]:
# --- 5.1 Comprobaciones automáticas por grupo ---
resumenes = {g: exp3.resumen_ejecucion(tabla[tabla["subconjunto"] == g]) for g in GRUPOS}
celda = lambda r: f"{r['k']} de {r['n']} {r['IC 95 %']}".strip() if r["n"] else "—"
tabla_resumen = pd.DataFrame({f"{g} ({(tabla['subconjunto'] == g).sum()})": resumenes[g].apply(celda, axis=1).values
                              for g in GRUPOS}, index=resumenes[GRUPOS[0]]["medida"].values)
display(tabla_resumen)

In [ ]:
# --- 5.2 Citas equivocadas por tipo de error y grupo, y ventanas con algún error de ese tipo ---
errores = tabla.groupby("subconjunto", observed=False)[COLUMNAS_ERROR].sum().T
errores.index = [c.replace("error: ", "") for c in errores.index]
ventanas_con = (tabla[COLUMNAS_ERROR] > 0).groupby(tabla["subconjunto"], observed=False).sum().T
ventanas_con.index = errores.index
errores.loc["total"] = errores.sum()
print("Citas equivocadas por tipo:")
display(errores)
print("Ventanas con algún error de ese tipo:")
display(ventanas_con)

In [ ]:
# --- 5.3 Lo que dice el modelo frente a tu referencia ---
NIVELES = ["ninguno", "bajo", "medio", "alto"]
ETIQUETAS = ["anomalia", "anomalia_mitigada", "normal", "dudosa"]
tablas_diagnostico = {}
for campo in ["origen", "evolucion_tipo", "impacto_video", "impacto_audio"]:
    t = pd.crosstab([tabla["subconjunto"], tabla["referencia"]], tabla[campo].fillna("(sin dato)"))
    if campo.startswith("impacto"):
        t = t[[c for c in NIVELES + ["(sin dato)"] if c in t.columns]]
    tablas_diagnostico[campo] = t
    print(campo)
    display(t)

## 6. Gráficas

- **6.1** Fidelidad automática por grupo: ventanas sin errores de cita y citas correctas (IC 95 %). La línea marca el 80 % de H₃.
- **6.2** Citas equivocadas por tipo de error y grupo.
- **6.3** Impacto en vídeo y en audio según tu referencia (prueba y desarrollo).
- **6.4** Origen que da el modelo según tu referencia.

Para pasar una gráfica a la memoria: botón de la cámara → PNG a 3 veces la resolución.

In [ ]:
# --- 6.1 Fidelidad automática por grupo ---
fig_fidelidad = go.Figure()
for medida, fila_resumen, color in (("Ventanas sin errores de cita", "Respuestas sin errores al citar la tabla", "#2a78d6"),
                                    ("Citas correctas", "Citas correctas", "#1baf7a")):
    k_n = [tuple(resumenes[g].set_index("medida").loc[fila_resumen, ["k", "n"]]) for g in GRUPOS]
    p = [k / n for k, n in k_n]
    ic = [est.intervalo_clopper_pearson(k, n) for k, n in k_n]
    fig_fidelidad.add_bar(x=GRUPOS, y=[100 * x for x in p], name=medida, marker_color=color,
                          text=[f"{k}/{n}" for k, n in k_n], textposition="outside", cliponaxis=False,
                          error_y=dict(type="data", symmetric=False, array=[100 * (s - x) for x, (i, s) in zip(p, ic)],
                                       arrayminus=[100 * (x - i) for x, (i, s) in zip(p, ic)], color="#52514e"))
fig_fidelidad.add_hline(y=80, line_dash="dash", line_color="#52514e", annotation_text="80 % (H₃)", annotation_position="top left")
est._estilo(fig_fidelidad, "Fidelidad automática de las explicaciones por grupo de ventanas (IC 95 %, Clopper-Pearson)", 480)
fig_fidelidad.update_layout(barmode="group", bargap=0.3)
fig_fidelidad.update_yaxes(range=[0, 115], title_text="%")
fig_fidelidad.show(config=est.CONFIG_PNG)

In [ ]:
# --- 6.2 Citas equivocadas por tipo de error y grupo ---
COLOR_TIPO = dict(zip(exp3.TIPOS_ERROR, ["#2a78d6", "#eb6834", "#f2b134", "#8a63d2", "#d2496a", "#52514e"]))
fig_tipos = go.Figure()
for tipo in exp3.TIPOS_ERROR:
    valores = [int(errores.loc[tipo, g]) for g in GRUPOS]
    if sum(valores):
        fig_tipos.add_bar(x=GRUPOS, y=valores, name=tipo, marker_color=COLOR_TIPO[tipo], text=valores, textposition="inside")
est._estilo(fig_tipos, "Citas equivocadas por tipo de error", 440)
fig_tipos.update_layout(barmode="stack", bargap=0.4)
fig_tipos.update_yaxes(title_text="Citas equivocadas")
fig_tipos.show(config=est.CONFIG_PNG)

In [ ]:
# --- 6.3 Impacto en vídeo y en audio según tu referencia (todas las ventanas) ---
COLOR_NIVEL = {"ninguno": "#c9c8c2", "bajo": "#1baf7a", "medio": "#f2b134", "alto": "#d2496a"}
fig_impacto = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=["Vídeo", "Audio"], horizontal_spacing=0.06)
for col, campo in enumerate(["impacto_video", "impacto_audio"], start=1):
    t = pd.crosstab(tabla["referencia"], tabla[campo]).reindex(index=ETIQUETAS, columns=NIVELES, fill_value=0)
    for nivel in NIVELES:
        fig_impacto.add_bar(x=t.index, y=t[nivel], name=nivel, marker_color=COLOR_NIVEL[nivel], legendgroup=nivel,
                            showlegend=(col == 1), text=t[nivel].where(t[nivel] > 0, ""), textposition="inside", row=1, col=col)
est._estilo(fig_impacto, "Impacto que da el modelo según tu referencia (prueba y desarrollo)", 460)
fig_impacto.update_layout(barmode="stack", bargap=0.3)
fig_impacto.update_yaxes(title_text="Ventanas", row=1, col=1)
fig_impacto.show(config=est.CONFIG_PNG)

In [ ]:
# --- 6.4 Origen que da el modelo según tu referencia ---
t = pd.crosstab(tabla["referencia"], tabla["origen"].fillna("(sin dato)")).reindex(index=ETIQUETAS, fill_value=0)
fig_origen = go.Figure()
for k, origen in enumerate(t.columns):
    fig_origen.add_bar(x=t.index, y=t[origen], name=origen, text=t[origen].where(t[origen] > 0, ""), textposition="inside",
                       marker_color=["#2a78d6", "#eb6834", "#1baf7a", "#8a63d2", "#52514e", "#f2b134"][k % 6])
est._estilo(fig_origen, "Origen del problema que da el modelo según tu referencia (prueba y desarrollo)", 440)
fig_origen.update_layout(barmode="stack", bargap=0.3)
fig_origen.update_yaxes(title_text="Ventanas")
fig_origen.show(config=est.CONFIG_PNG)

## 7. Resumen

In [ ]:
# --- 7. Resumen en texto de las cifras principales ---
def fila(g, medida):
    r = resumenes[g].set_index("medida").loc[medida]
    return f"{r['k']} de {r['n']} {r['IC 95 %']}"

a = tabla[tabla["subconjunto"] == "alertas"]
sin_anomalia = tabla[tabla["referencia"].isin(["normal", "anomalia_mitigada"])]
print(f"Ejecuciones: {EJECUCION_PRUEBA} (prueba) y {EJECUCION_DESARROLLO} (desarrollo) · prompt {HUELLA}\n")
print("Formato:", ", ".join(f"{g}: {fila(g, 'Respuestas con JSON válido')} JSON válidos" for g in GRUPOS))
print(f"Fidelidad automática en las 20 alertas: {fila('alertas', 'Respuestas sin errores al citar la tabla')} sin errores de "
      f"cita; {fila('alertas', 'Citas correctas')} citas correctas; "
      f"{fila('alertas', 'Respuestas sin ninguna cita comprobable')} sin ninguna cita comprobable.")
print("Valores calculados bien copiados:",
      ", ".join(f"{g}: {fila(g, 'Valores calculados copiados correctamente')}" for g in GRUPOS))
print("Citas equivocadas por tipo (todas las ventanas):",
      ", ".join(f"{t}: {int(errores.loc[t].sum())}" for t in exp3.TIPOS_ERROR if errores.loc[t].sum()))
print("Nivel de impacto coherente con la escala (medida aparte de H₃):", ", ".join(
    f"{g}: vídeo {fila(g, 'Nivel de impacto en vídeo coherente con la escala')}, "
    f"audio {fila(g, 'Nivel de impacto en audio coherente con la escala')}" for g in GRUPOS))
print(f"Impacto en vídeo en las alertas: {a['impacto_video'].value_counts().to_dict()}")
print(f"Impacto en vídeo en las mitigadas y normales ({len(sin_anomalia)}): {sin_anomalia['impacto_video'].value_counts().to_dict()}")
print(f"Origen en las mitigadas y normales: {sin_anomalia['origen'].value_counts().to_dict()}")
print(f"Evolución (todas): {tabla['evolucion_tipo'].value_counts().to_dict()}")

## 8. Informe HTML con desplegable (privado)

Para revisar y puntuar cada ventana: un desplegable y flechas para pasar de una a otra; por ventana, la gráfica y la tabla con los datos que vio el modelo (con los nombres reales), las comprobaciones con el tipo de cada error, el impacto frente a la escala y la respuesta por campos. Se guarda en `privado/experimento_3/informes/`: descárgalo de Drive y ábrelo en el navegador (funciona sin conexión).

In [ ]:
# --- 8. Informe HTML (prueba y desarrollo), en la carpeta privada ---
for base, nombre in ((EJECUCION_PRUEBA, "prueba"), (EJECUCION_DESARROLLO, "desarrollo")):
    t = tabla[tabla["ejecucion"] == base]
    registro = {**REGISTROS[base], "llamadas": [c for c in REGISTROS[base]["llamadas"] if c["id"] in set(t["id"])]}
    registro["llamadas"].sort(key=lambda c: (SUBCONJUNTO[c["id"]] != "alertas", c["id"]))   # las alertas primero
    ruta = exp3.informe_html(registro, t, prompts, fuente, VENTANAS,
                             f"{RUTA_INFORMES}/informe_{nombre}_{datetime.now():%Y%m%d_%H%M}.html",
                             f"Experimento 3 · {nombre} · {base}")
    print("Informe (privado):", os.path.basename(ruta))

## 9. Puntuaciones y H₃

Rúbrica (adaptada de la Tabla 4 de Y. Liu, Tao, Meng, Wang et al., 2024): **utilidad** y **legibilidad** de 1 a 5 y **error_fidelidad** = 1 si la explicación afirma algo que la tabla contradice o que no se puede comprobar con ella (un valor, un reporte, un origen o una tendencia), aunque no lo escriba como cita. En `error_fidelidad` solo se anotan los errores que no detecta la comprobación automática.

**Regla de utilidad** (para aplicar el mismo criterio a todas): base 4 si el origen es correcto o defendible con los datos; 3 si es equivocado sin respaldo; −1 si es falso un hecho principal (pérdida, recuperación y mecanismo, inicio o fases) o la conclusión se apoya en una afirmación falsa; no restan un valor puntual, un rango o un pico secundario; 5 solo sin errores de fidelidad.

- **9.1 (una sola vez):** copia tus puntuaciones de las 20 alertas, guardadas por el notebook de trabajo, a `privado/experimento_3/evaluacion/` (con tus comentarios). La carpeta de origen solo se lee.
- **9.2:** H₃ con intervalos y la gráfica de puntuaciones. Solo usa las cifras (sin comentarios).

In [ ]:
# --- 9.1 Puntuaciones de las 20 alertas (copia, una sola vez, a la carpeta privada) ---
ORIGEN_PUNTUACIONES = "/content/drive/MyDrive/TFM_VIU/export_colab_split_estratificado/experimento_3/evaluacion"   # solo lectura

nombre = f"puntuaciones_{EJECUCION_PRUEBA}.json"
os.makedirs(RUTA_EVALUACION_PRIVADA, exist_ok=True)
if not os.path.exists(f"{RUTA_EVALUACION_PRIVADA}/{nombre}"):
    shutil.copy2(f"{ORIGEN_PUNTUACIONES}/{nombre}", f"{RUTA_EVALUACION_PRIVADA}/{nombre}")
    print("Copiado:", nombre)
completas = json.load(open(f"{RUTA_EVALUACION_PRIVADA}/{nombre}", encoding="utf-8"))
# Solo las cifras: los comentarios son notas del autor y se quedan en la carpeta privada
PUNTUACIONES = {i: {k: p[k] for k in ("utilidad", "legibilidad", "error_fidelidad")} for i, p in completas.items()}
alertas_ids = set(tabla.loc[tabla["subconjunto"] == "alertas", "id"])
print(f"{len(PUNTUACIONES)} puntuaciones · alertas sin puntuar: {sorted(alertas_ids - set(PUNTUACIONES)) or 'ninguna'}")

In [ ]:
# --- 9.2 H₃ en las 20 alertas: medias con IC 95 % (t de Student) y proporción sin errores de fidelidad (Clopper-Pearson) ---
por_ventana, resumen_h3 = exp3.evaluar_h3(PUNTUACIONES, tabla[tabla["subconjunto"] == "alertas"])
display(resumen_h3)
display(por_ventana)
fig_puntuaciones = exp3.grafica_puntuaciones(por_ventana)
fig_puntuaciones.show(config=est.CONFIG_PNG)

## 10. Guardado

En `resultados/experimento_3/`: las tablas en CSV, las gráficas en HTML (se abren sin conexión) y un JSON con todo lo anterior y una fila por ventana. Solo cifras, categorías y nombres cortos: es lo que se puede compartir.

In [ ]:
# --- 10. Cifras (sin textos) en resultados/experimento_3 ---
carpeta = f"{RUTA_EXPERIMENTO_3}/resultados_{datetime.now():%Y%m%d_%H%M}"
os.makedirs(carpeta, exist_ok=True)
NUMERICAS = [c for c in tabla.columns if c not in exp3.COLUMNAS_TEXTO]   # sin los errores de cita con nombres reales
tablas = {"por_ventana": tabla[NUMERICAS], "comprobaciones_por_grupo": tabla_resumen.reset_index(names="medida"),
          "tipos_de_error": errores.reset_index(names="tipo"), "h3": resumen_h3, "h3_por_ventana": por_ventana,
          **{f"diagnostico_{k}": v.reset_index() for k, v in tablas_diagnostico.items()}}
for nombre_tabla, t in tablas.items():
    t.to_csv(f"{carpeta}/{nombre_tabla}.csv", index=False)
figuras = {"fidelidad_por_grupo": fig_fidelidad, "tipos_de_error": fig_tipos, "impacto_segun_referencia": fig_impacto,
           "origen_segun_referencia": fig_origen, "puntuaciones": fig_puntuaciones}
for nombre_figura, fig in figuras.items():
    fig.write_html(f"{carpeta}/{nombre_figura}.html", include_plotlyjs=True, config=est.CONFIG_PNG)
registros = lambda t: json.loads(t.to_json(orient="records"))   # NaN -> null y tipos de numpy -> JSON
resumen = {"fecha": datetime.now().strftime("%Y-%m-%d %H:%M"), "huella_prompt": HUELLA, "variante": VARIANTE,
           "ejecuciones": {"prueba": EJECUCION_PRUEBA, "desarrollo": EJECUCION_DESARROLLO},
           **{k: registros(t) for k, t in tablas.items()}}
with open(f"{carpeta}/resumen_experimento_3.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=1)
print("Guardado en:", carpeta)
print("Ficheros:", sorted(os.listdir(carpeta)))